In [13]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold

In [14]:
train = pd.read_csv('csv/train.csv')
test = pd.read_csv('csv/test.csv')
sample_submission = pd.read_csv('csv/sample_submission.csv')

In [15]:
#특성과 타겟 변수 분리
train = train.drop(columns=['ID'], axis = 1)
test = test.drop(columns=['ID'], axis = 1)

In [16]:
# 설립연도 타입 변환 (int -> object)
train['설립연도'] =train['설립연도'].astype('object')
test['설립연도'] =test['설립연도'].astype('object')

category_features = ['설립연도','국가','분야','투자단계','기업가치(백억원)']
numeric_features = ['직원 수','고객수(백만명)','총 투자금(억원)','연매출(억원)','SNS 팔로워 수(백만명)']
bool_features = ['인수여부','상장여부']

# LabelEncoder 객체를 각 범주형 feature별로 따로 저장하여 사용
encoders = {}

# 범주형 데이터를 encoding
for feature in category_features:
    encoders[feature] = LabelEncoder()
    train[feature] = train[feature].fillna('Missing')
    test[feature] = test[feature].fillna('Missing')
    train[feature] = encoders[feature].fit_transform(train[feature])
    test[feature] = encoders[feature].transform(test[feature])

# 불리언 값을 0과 1로 변환 ('Yes' → 1, 'No' → 0 으로 변환)
bool_map = {'Yes': 1, 'No': 0}

for feature in bool_features:
    train[feature] = train[feature].map(bool_map)
    test[feature] = test[feature].map(bool_map)

# 수치형 변수 결측치를 평균값으로 대체
for feature in numeric_features:
    mean_value = train[feature].mean()
    train[feature] = train[feature].fillna(mean_value)
    test[feature] = test[feature].fillna(mean_value)


/var/folders/ss/j3gw42tn6hj8yhjfpvq0shb00000gn/T/ipykernel_40704/3938303409.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train[feature] = train[feature].fillna('Missing')
/var/folders/ss/j3gw42tn6hj8yhjfpvq0shb00000gn/T/ipykernel_40704/3938303409.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test[feature] = test[feature].fillna('Missing')


In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split

# X: features, y: soft labels (확률 값)
X_train, X_test, y_train, y_test = train_test_split(train.drop(columns='성공확률',axis=1), train['성공확률'], test_size=0.2, random_state=42)

# 회귀 모델 예시
model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.01,
    max_depth=5,
    subsample=0.8
)
model.fit(X_train, y_train)

# 테스트 데이터에 대한 확률 예측
y_pred = model.predict(X_test)

In [18]:
from sklearn.model_selection import GridSearchCV

model = GradientBoostingRegressor()

param_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5],
    'subsample': [0.6, 0.8, 1.0],
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,  # 5-fold 교차검증
    scoring='neg_mean_squared_error',  # 회귀라서 MSE 기준
    n_jobs=-1,  # 가능한 모든 CPU 사용
    verbose=1
)

# 학습
grid_search.fit(X_train, y_train)

# 최적의 파라미터 조합
print("Best parameters:", grid_search.best_params_)
print("Best score (MSE):", -grid_search.best_score_)

Fitting 5 folds for each of 54 candidates, totalling 270 fits


KeyboardInterrupt: 

In [19]:
# 회귀 모델 파라미터 조정
model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.01,
    max_depth=5,
    subsample=0.8
)
model.fit(X_train, y_train)

# 테스트 데이터에 대한 확률 예측
y_pred = model.predict(X_test)

In [20]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test, y_pred)
print(mse)

0.05826052369870615


In [21]:
print(y_pred)

[0.53104799 0.5460975  0.52318137 0.53810442 0.5379187  0.53098098
 0.53266821 0.54344132 0.53094214 0.53143021 0.53009486 0.53431962
 0.53607404 0.53895408 0.53424885 0.54850085 0.53832069 0.52307011
 0.54168451 0.54539553 0.56950793 0.53652262 0.54166081 0.55717353
 0.55209041 0.54189012 0.53738123 0.5528145  0.53906201 0.52284217
 0.54014413 0.51118977 0.53707815 0.50531669 0.53049213 0.5265973
 0.53526928 0.54249765 0.53927997 0.54860799 0.53672292 0.55101099
 0.52793828 0.55121825 0.54629008 0.50637873 0.50249343 0.53727654
 0.51308949 0.53934237 0.54323281 0.53461435 0.52057675 0.54403236
 0.52583057 0.51981845 0.52641928 0.52807291 0.53303907 0.54976244
 0.56157815 0.51862444 0.51558492 0.53570281 0.52457546 0.56757107
 0.53305532 0.53492963 0.56995131 0.54778845 0.53667613 0.54408636
 0.53877173 0.53319832 0.52723065 0.51452153 0.55734602 0.52193482
 0.53758416 0.5703391  0.53789406 0.54011377 0.55380652 0.52072581
 0.53331166 0.52042642 0.56089422 0.49372795 0.55996735 0.49787

In [22]:
print(np.var(y_pred))

0.00026690545989893


In [23]:
print(y_test)

4296    0.3
3530    0.5
1228    0.8
911     0.5
2909    0.1
       ... 
3279    0.5
1485    0.3
1788    0.8
1609    0.8
2662    0.9
Name: 성공확률, Length: 876, dtype: float64


In [24]:
print(y_test-y_pred)

4296   -0.231048
3530   -0.046097
1228    0.276819
911    -0.038104
2909   -0.437919
          ...   
3279   -0.050849
1485   -0.238731
1788    0.303942
1609    0.262147
2662    0.343692
Name: 성공확률, Length: 876, dtype: float64


In [25]:
#제출
model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.01,
    max_depth=5,
    subsample=0.8
)
model.fit(train.drop(columns=['성공확률']), train['성공확률'])

pred = model.predict(test)

sample_submission['성공확률'] = pred
sample_submission.to_csv('./baseline_submission.csv', index = False, encoding = 'utf-8-sig')

In [26]:
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

print("Train MSE:", mean_squared_error(y_train, train_pred))
print("Test MSE:", mean_squared_error(y_test, test_pred))

Train MSE: 0.05470534672740005
Test MSE: 0.054380216044653006


In [27]:
model.score(X_train,y_train)

0.06512874724645645